# 逐次予測

前処理済みのデータを用いてLigthGBMを用いてモデルを構築し、評価する

- 目的変数: `price_actual`
- モデル: LightGBM
- 評価指標: RMSE
- ハイパーパラメータチューニング: ベイズ最適化

## 1. ライブラリのインポートとデータ読み込み

In [1]:
# 自動ローディング
%load_ext autoreload
%autoreload 2

In [3]:
# アクティベート
!source ../../.venv/bin/activate

In [4]:
# LightGBM特有のエラー対策
#!brew install libomp
#!pip uninstall lightgbm
#!pip install lightgbm

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# データ読み込み用の関数をインポート
import sys, os
sys.path.append(os.pardir)  # 親ディレクトリのファイルをインポートするための設定
from src.modeling import train_and_predict

# データディレクトリ
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
SUBMISSION = DATA_DIR / 'submission'

os.makedirs(SUBMISSION, exist_ok=True)

print(DATA_DIR)
# 前処理済みデータの読み込み
train = pd.read_csv(DATA_DIR / 'train_processed.csv')
test = pd.read_csv(DATA_DIR / 'test_processed.csv')

print('train shape:', train.shape)
print('test shape:', test.shape)

/Users/m0122wt/Desktop/02.プライベート/01.ノウハウ/07.データ分析/notebook/signate_smbc_202506/data
train shape: (26280, 94)
test shape: (8760, 93)


## 2. 特徴量・目的変数の設定

In [14]:
# 目的変数
target_col = 'price_actual'

# 説明変数（目的変数とtime列以外）
drop_cols = ['time', target_col] if target_col in train.columns else ['time']
feature_cols = [col for col in train.columns if col not in drop_cols]

X = train[feature_cols]
y = train[target_col] if target_col in train.columns else train.iloc[:, -1]  # 念のため

print('Features:', feature_cols)
print('Target:', target_col)
print('X shape:', X.shape)
print('y shape:', y.shape)

Features: ['generation_biomass', 'generation_fossil_brown_coal/lignite', 'generation_fossil_gas', 'generation_fossil_hard_coal', 'generation_fossil_oil', 'generation_hydro_pumped_storage_consumption', 'generation_hydro_run_of_river_and_poundage', 'generation_hydro_water_reservoir', 'generation_nuclear', 'generation_other', 'generation_other_renewable', 'generation_solar', 'generation_waste', 'generation_wind_onshore', 'total_load_actual', 'valencia_pressure', 'valencia_humidity', 'valencia_wind_speed', 'valencia_wind_deg', 'valencia_rain_1h', 'valencia_rain_3h', 'valencia_snow_3h', 'valencia_clouds_all', 'madrid_pressure', 'madrid_humidity', 'madrid_wind_speed', 'madrid_wind_deg', 'madrid_rain_1h', 'madrid_rain_3h', 'madrid_snow_3h', 'madrid_clouds_all', 'bilbao_pressure', 'bilbao_humidity', 'bilbao_wind_speed', 'bilbao_wind_deg', 'bilbao_rain_1h', 'bilbao_rain_3h', 'bilbao_snow_3h', 'bilbao_clouds_all', 'barcelona_pressure', 'barcelona_humidity', 'barcelona_wind_speed', 'barcelona_win

In [15]:
# 学習とテストデータでカラムの構成に違いがないか確認
diff_features = set(train.columns) - set(test.columns)
print("差分があるカラム:")
print(sorted(list(diff_features)))

差分があるカラム:
['price_actual']


## 3. 学習・検証

In [16]:
%%time
filename = 'submission_lgbm'

# 通常の予測
model, predictions = train_and_predict(
    model_type='lightgbm',
    train_df=train,
    test_df=test,
    target_col='price_actual',
    optimize=True,
    sequential=False,  # 通常予測
    output_path=SUBMISSION,
    filename=filename
)

[I 2025-06-27 15:43:17,721] A new study created in memory with name: no-name-37f64ff8-dc4f-4d4c-aba4-65a48c3e24ce


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[41]	valid_0's rmse: 2.66805
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[78]	valid_0's rmse: 3.33617
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[90]	valid_0's rmse: 2.21106
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[95]	valid_0's rmse: 2.41199
Training until validation scores don't improve for 50 rounds


[I 2025-06-27 15:43:21,135] Trial 0 finished with value: 2.5765884159185872 and parameters: {'num_leaves': 22, 'learning_rate': 0.13947744114085783, 'feature_fraction': 0.9784154511629671, 'bagging_fraction': 0.985513595810184, 'bagging_freq': 10, 'min_child_samples': 70}. Best is trial 0 with value: 2.5765884159185872.


Early stopping, best iteration is:
[115]	valid_0's rmse: 2.25568
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[40]	valid_0's rmse: 3.22525
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[51]	valid_0's rmse: 4.77397
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[33]	valid_0's rmse: 2.84041
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[82]	valid_0's rmse: 3.38193
Training until validation scores don't improve for 50 rounds


[I 2025-06-27 15:43:27,265] Trial 1 finished with value: 3.330474638338366 and parameters: {'num_leaves': 74, 'learning_rate': 0.17955205713221473, 'feature_fraction': 0.636631223971213, 'bagging_fraction': 0.9230386829958792, 'bagging_freq': 10, 'min_child_samples': 68}. Best is trial 0 with value: 2.5765884159185872.


...

Early stopping, best iteration is:
[256]	valid_0's rmse: 2.22093
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[49]	valid_0's rmse: 2.63436
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[99]	valid_0's rmse: 3.40402
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[53]	valid_0's rmse: 2.17378
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[137]	valid_0's rmse: 2.35886
Training until validation scores don't improve for 50 rounds


[I 2025-06-27 15:57:47,622] Trial 98 finished with value: 2.5603042999006442 and parameters: {'num_leaves': 27, 'learning_rate': 0.09518737326095723, 'feature_fraction': 0.9288507309465541, 'bagging_fraction': 0.9077880116062675, 'bagging_freq': 9, 'min_child_samples': 25}. Best is trial 92 with value: 2.517122779389563.


Early stopping, best iteration is:
[259]	valid_0's rmse: 2.2305
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[130]	valid_0's rmse: 2.67705
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[168]	valid_0's rmse: 4.01087
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[95]	valid_0's rmse: 2.33688
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[216]	valid_0's rmse: 2.59993
Training until validation scores don't improve for 50 rounds


[I 2025-06-27 15:58:10,734] Trial 99 finished with value: 2.7738060413636303 and parameters: {'num_leaves': 99, 'learning_rate': 0.04335762299661873, 'feature_fraction': 0.8836453403480073, 'bagging_fraction': 0.8916517614568094, 'bagging_freq': 9, 'min_child_samples': 29}. Best is trial 92 with value: 2.517122779389563.


Early stopping, best iteration is:
[245]	valid_0's rmse: 2.24429
LIGHTGBM：ベイズ最適化の結果
Best params: {'num_leaves': 20, 'learning_rate': 0.07056499143559153, 'feature_fraction': 0.930666633751073, 'bagging_fraction': 0.8764523692423547, 'bagging_freq': 9, 'min_child_samples': 35, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'verbose': -1, 'random_state': 42}
Best CV RMSE: 2.517122779389563
Selected features: 73/92
Submission saved: /Users/m0122wt/Desktop/02.プライベート/01.ノウハウ/07.データ分析/notebook/signate_smbc_202506/data/submission/submission_lgbm_20250627_155820.csv


### 履歴
___
LIGHTGBM：ベイズ最適化の結果
Best params: {'num_leaves': 20, 'learning_rate': 0.07056499143559153, 'feature_fraction': 0.930666633751073, 'bagging_fraction': 0.8764523692423547, 'bagging_freq': 9, 'min_child_samples': 35, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'verbose': -1, 'random_state': 42}  
Best CV RMSE: 2.517122779389563  
Selected features: 73/92  
___

In [ ]:
predictions

In [17]:
%%time

filename_sequential = 'submission_lgbm_sequential'

# 逐次予測（ラグ特徴量を考慮）
model_sequential, predictions_sequential = train_and_predict(
    model_type='lightgbm',
    train_df=train,
    test_df=test,
    target_col='price_actual',
    optimize=True,
    sequential=True,  # 逐次予測
    output_path=SUBMISSION,
    filename=filename_sequential
)

[I 2025-06-27 15:58:21,009] A new study created in memory with name: no-name-7d32502b-c512-444a-905a-365a68b92497


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[19]	valid_0's rmse: 2.79812
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[30]	valid_0's rmse: 3.61769
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[22]	valid_0's rmse: 2.29793
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[46]	valid_0's rmse: 2.50405
Training until validation scores don't improve for 50 rounds


[I 2025-06-27 15:58:26,325] Trial 0 finished with value: 2.713237283027849 and parameters: {'num_leaves': 65, 'learning_rate': 0.22388646353637037, 'feature_fraction': 0.9079489693500611, 'bagging_fraction': 0.698992201254674, 'bagging_freq': 6, 'min_child_samples': 34}. Best is trial 0 with value: 2.713237283027849.


Early stopping, best iteration is:
[31]	valid_0's rmse: 2.3484
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[95]	valid_0's rmse: 2.68755
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[220]	valid_0's rmse: 3.58464
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[162]	valid_0's rmse: 2.28981
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[229]	valid_0's rmse: 2.41406
Training until validation scores don't improve for 50 rounds


[I 2025-06-27 15:58:35,546] Trial 1 finished with value: 2.6452906590201737 and parameters: {'num_leaves': 32, 'learning_rate': 0.05447631852200481, 'feature_fraction': 0.809733012145131, 'bagging_fraction': 0.6238454977002927, 'bagging_freq': 6, 'min_child_samples': 29}. Best is trial 1 with value: 2.6452906590201737.


...

Early stopping, best iteration is:
[353]	valid_0's rmse: 2.23844
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[66]	valid_0's rmse: 2.6544
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[109]	valid_0's rmse: 3.22891
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[87]	valid_0's rmse: 2.10699
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[160]	valid_0's rmse: 2.35573
Training until validation scores don't improve for 50 rounds


[I 2025-06-27 16:15:52,245] Trial 98 finished with value: 2.51734282613965 and parameters: {'num_leaves': 21, 'learning_rate': 0.0926973092334304, 'feature_fraction': 0.9533804854356259, 'bagging_fraction': 0.9723501183105292, 'bagging_freq': 10, 'min_child_samples': 43}. Best is trial 98 with value: 2.51734282613965.


Early stopping, best iteration is:
[355]	valid_0's rmse: 2.24069
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[51]	valid_0's rmse: 2.65312
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[111]	valid_0's rmse: 3.28601
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[114]	valid_0's rmse: 2.14081
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[152]	valid_0's rmse: 2.35603
Training until validation scores don't improve for 50 rounds


[I 2025-06-27 16:15:57,288] Trial 99 finished with value: 2.5388414350184747 and parameters: {'num_leaves': 20, 'learning_rate': 0.1000859161555205, 'feature_fraction': 0.9300816780994856, 'bagging_fraction': 0.9668033546526921, 'bagging_freq': 10, 'min_child_samples': 43}. Best is trial 98 with value: 2.51734282613965.


Early stopping, best iteration is:
[278]	valid_0's rmse: 2.25823
LIGHTGBM：ベイズ最適化の結果
Best params: {'num_leaves': 21, 'learning_rate': 0.0926973092334304, 'feature_fraction': 0.9533804854356259, 'bagging_fraction': 0.9723501183105292, 'bagging_freq': 10, 'min_child_samples': 43, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'verbose': -1, 'random_state': 42}
Best CV RMSE: 2.51734282613965
Selected features: 73/92
Submission saved: /Users/m0122wt/Desktop/02.プライベート/01.ノウハウ/07.データ分析/notebook/signate_smbc_202506/data/submission/submission_lgbm_sequential_20250627_161613.csv
CPU times: user 25min 49s, sys: 18min 50s, total: 44min 40s
Wall time: 17min 52s


### 履歴
___
LIGHTGBM：ベイズ最適化の結果
Best params: {'num_leaves': 21, 'learning_rate': 0.0926973092334304, 'feature_fraction': 0.9533804854356259, 'bagging_fraction': 0.9723501183105292, 'bagging_freq': 10, 'min_child_samples': 43, 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'verbose': -1, 'random_state': 42}  
Best CV RMSE: 2.51734282613965  
Selected features: 73/92
___

In [18]:
predictions_sequential

array([21.68868437, 21.68936646, 22.92291243, ..., 78.370438  ,
       73.02906532, 68.37928134])